# Transformer Architecture

This notebook explores the Transformer architecture introduced in "Attention Is All You Need" (Vaswani et al., 2017). We will:

1. Visualize the high-level Transformer architecture with matplotlib
2. Implement **scaled dot-product attention** from scratch with NumPy
3. Walk through self-attention with a small 4-token example
4. Visualize attention weights as a heatmap
5. Explain multi-head attention

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_style('whitegrid')
np.random.seed(42)

---
## 1. Transformer Architecture Diagram

The Transformer consists of an **encoder** and a **decoder**, each made of stacked layers. The key innovation is the **self-attention mechanism** that replaces recurrence entirely.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
ax.set_xlim(0, 12)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Transformer Architecture (Simplified)', fontsize=16, fontweight='bold', pad=20)

def draw_box(ax, x, y, w, h, text, color='lightblue', fontsize=9):
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.1",
                                     facecolor=color, edgecolor='black', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=fontsize, fontweight='bold')

def draw_arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

# Encoder side (left)
ax.text(3, 9.5, 'ENCODER', ha='center', fontsize=13, fontweight='bold', color='navy')

draw_box(ax, 1.5, 0.5, 3, 0.7, 'Input Embedding\n+ Positional Encoding', color='#FFD700', fontsize=8)
draw_arrow(ax, 3, 1.2, 3, 1.8)

draw_box(ax, 1.5, 1.8, 3, 0.7, 'Multi-Head\nSelf-Attention', color='#87CEEB', fontsize=8)
draw_arrow(ax, 3, 2.5, 3, 3.1)

draw_box(ax, 1.5, 3.1, 3, 0.5, 'Add & Norm', color='#98FB98', fontsize=8)
draw_arrow(ax, 3, 3.6, 3, 4.2)

draw_box(ax, 1.5, 4.2, 3, 0.7, 'Feed-Forward\nNetwork', color='#DDA0DD', fontsize=8)
draw_arrow(ax, 3, 4.9, 3, 5.5)

draw_box(ax, 1.5, 5.5, 3, 0.5, 'Add & Norm', color='#98FB98', fontsize=8)

# Nx label
ax.text(0.8, 3.5, 'N x', fontsize=14, fontweight='bold', color='red',
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='red'))

# Decoder side (right)
ax.text(9, 9.5, 'DECODER', ha='center', fontsize=13, fontweight='bold', color='darkred')

draw_box(ax, 7.5, 0.5, 3, 0.7, 'Output Embedding\n+ Positional Encoding', color='#FFD700', fontsize=8)
draw_arrow(ax, 9, 1.2, 9, 1.8)

draw_box(ax, 7.5, 1.8, 3, 0.7, 'Masked Multi-Head\nSelf-Attention', color='#87CEEB', fontsize=8)
draw_arrow(ax, 9, 2.5, 9, 3.1)

draw_box(ax, 7.5, 3.1, 3, 0.5, 'Add & Norm', color='#98FB98', fontsize=8)
draw_arrow(ax, 9, 3.6, 9, 4.2)

draw_box(ax, 7.5, 4.2, 3, 0.7, 'Multi-Head\nCross-Attention', color='#F0E68C', fontsize=8)
draw_arrow(ax, 9, 4.9, 9, 5.5)

draw_box(ax, 7.5, 5.5, 3, 0.5, 'Add & Norm', color='#98FB98', fontsize=8)
draw_arrow(ax, 9, 6.0, 9, 6.5)

draw_box(ax, 7.5, 6.5, 3, 0.7, 'Feed-Forward\nNetwork', color='#DDA0DD', fontsize=8)
draw_arrow(ax, 9, 7.2, 9, 7.7)

draw_box(ax, 7.5, 7.7, 3, 0.5, 'Add & Norm', color='#98FB98', fontsize=8)
draw_arrow(ax, 9, 8.2, 9, 8.6)

draw_box(ax, 7.5, 8.6, 3, 0.6, 'Linear + Softmax', color='#FFA07A', fontsize=8)

# Cross-attention arrow from encoder to decoder
draw_arrow(ax, 4.5, 5.75, 7.5, 4.55)

# Nx label
ax.text(6.8, 5.0, 'N x', fontsize=14, fontweight='bold', color='red',
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='red'))

plt.tight_layout()
plt.show()

---
## 2. Scaled Dot-Product Attention (From Scratch)

The core equation:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

Where:
- **Q** (Query): what we're looking for
- **K** (Key): what we match against
- **V** (Value): what we actually retrieve
- **d_k**: dimension of keys (for scaling)

In [ ]:
def softmax(x, axis=-1):
    """Numerically stable softmax."""
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)


def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Compute scaled dot-product attention.
    
    Args:
        Q: Query matrix (seq_len_q, d_k)
        K: Key matrix (seq_len_k, d_k)
        V: Value matrix (seq_len_k, d_v)
        mask: Optional mask to prevent attending to certain positions
    
    Returns:
        output: Weighted sum of values (seq_len_q, d_v)
        attention_weights: Attention weights (seq_len_q, seq_len_k)
    """
    d_k = K.shape[-1]
    
    # Step 1: Compute dot products Q * K^T
    scores = Q @ K.T
    print(f"  Raw scores (Q @ K^T) shape: {scores.shape}")
    
    # Step 2: Scale by sqrt(d_k)
    scores = scores / np.sqrt(d_k)
    print(f"  Scaled scores (/ sqrt({d_k})): min={scores.min():.3f}, max={scores.max():.3f}")
    
    # Step 3: Apply mask (if provided)
    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)
    
    # Step 4: Softmax to get attention weights
    attention_weights = softmax(scores)
    print(f"  Attention weights (each row sums to 1):")
    print(f"  Row sums: {attention_weights.sum(axis=-1)}")
    
    # Step 5: Weighted sum of values
    output = attention_weights @ V
    
    return output, attention_weights


print("Scaled dot-product attention function defined.")

---
## 3. Self-Attention Walkthrough (4 Tokens)

Let's walk through self-attention with a concrete example: the sentence **"The cat sat down"** (4 tokens).

Each token is represented by a small embedding vector, and we compute Q, K, V by multiplying with learned weight matrices.

In [ ]:
# Define our 4-token example
tokens = ["The", "cat", "sat", "down"]
seq_len = len(tokens)
d_model = 8   # embedding dimension
d_k = 4       # key/query dimension
d_v = 4       # value dimension

# Simulated token embeddings (in practice, these are learned)
X = np.random.randn(seq_len, d_model)
print(f"Input embeddings X shape: {X.shape}")
print(f"  (4 tokens, each with {d_model}-dim embedding)\n")

# Learned weight matrices (in practice, these are parameters)
W_Q = np.random.randn(d_model, d_k) * 0.1
W_K = np.random.randn(d_model, d_k) * 0.1
W_V = np.random.randn(d_model, d_v) * 0.1

# Compute Q, K, V
Q = X @ W_Q
K = X @ W_K
V = X @ W_V

print(f"Q (queries) shape: {Q.shape}")
print(f"K (keys) shape:    {K.shape}")
print(f"V (values) shape:  {V.shape}")
print()

# Run attention
print("Computing attention:")
output, attn_weights = scaled_dot_product_attention(Q, K, V)

print(f"\nOutput shape: {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")

### Interpreting the Attention Weights

Each row `i` in the attention weight matrix shows how much token `i` attends to every other token. The weights sum to 1 across each row (due to softmax).

In [ ]:
print("Attention weight matrix:")
print("(row = query token, column = key token)\n")

# Pretty print
header = "         " + "  ".join(f"{t:>6}" for t in tokens)
print(header)
for i, token in enumerate(tokens):
    row = "  ".join(f"{w:6.3f}" for w in attn_weights[i])
    print(f"{token:>6}   {row}")

---
## 4. Visualize Attention Weights as Heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap of attention weights
sns.heatmap(attn_weights, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=tokens, yticklabels=tokens, ax=axes[0],
            vmin=0, vmax=1, linewidths=0.5)
axes[0].set_title('Self-Attention Weights', fontsize=14)
axes[0].set_xlabel('Key (attending to)', fontsize=11)
axes[0].set_ylabel('Query (attending from)', fontsize=11)

# Causal (decoder-style) mask - lower triangular
causal_mask = np.tril(np.ones((seq_len, seq_len)))
print("Causal mask (decoder):")
print(causal_mask)

output_masked, attn_weights_masked = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

sns.heatmap(attn_weights_masked, annot=True, fmt='.3f', cmap='Reds',
            xticklabels=tokens, yticklabels=tokens, ax=axes[1],
            vmin=0, vmax=1, linewidths=0.5)
axes[1].set_title('Masked Self-Attention (Causal)', fontsize=14)
axes[1].set_xlabel('Key (attending to)', fontsize=11)
axes[1].set_ylabel('Query (attending from)', fontsize=11)

plt.tight_layout()
plt.show()

print("\nNote: In the causal mask (right), each token can only attend to itself")
print("and previous tokens. Future positions are masked out (set to ~0).")

---
## 5. Multi-Head Attention

Instead of performing a single attention function, the Transformer uses **multiple attention heads** in parallel. Each head can learn to focus on different types of relationships.

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h) W^O$$

where $\text{head}_i = \text{Attention}(Q W_i^Q, K W_i^K, V W_i^V)$

In [ ]:
def multi_head_attention(X, num_heads, d_model, d_k, d_v):
    """
    Multi-head attention with random weight matrices.
    
    Args:
        X: Input embeddings (seq_len, d_model)
        num_heads: Number of attention heads
        d_model: Embedding dimension
        d_k: Key/query dimension per head
        d_v: Value dimension per head
    
    Returns:
        output: Multi-head attention output
        all_weights: Attention weights from each head
    """
    all_weights = []
    head_outputs = []
    
    for h in range(num_heads):
        # Each head has its own weight matrices
        W_Q_h = np.random.randn(d_model, d_k) * 0.1
        W_K_h = np.random.randn(d_model, d_k) * 0.1
        W_V_h = np.random.randn(d_model, d_v) * 0.1
        
        Q_h = X @ W_Q_h
        K_h = X @ W_K_h
        V_h = X @ W_V_h
        
        # Compute attention for this head (suppress prints)
        scores = (Q_h @ K_h.T) / np.sqrt(d_k)
        weights = softmax(scores)
        head_out = weights @ V_h
        
        head_outputs.append(head_out)
        all_weights.append(weights)
    
    # Concatenate all heads
    concat = np.concatenate(head_outputs, axis=-1)
    
    # Final linear projection
    W_O = np.random.randn(num_heads * d_v, d_model) * 0.1
    output = concat @ W_O
    
    return output, all_weights


# Run multi-head attention with 4 heads
num_heads = 4
mha_output, head_weights = multi_head_attention(X, num_heads=num_heads, 
                                                  d_model=d_model, d_k=3, d_v=3)

print(f"Multi-head attention output shape: {mha_output.shape}")
print(f"Number of heads: {num_heads}")
print(f"Each head produces attention weights of shape: {head_weights[0].shape}")

In [ ]:
# Visualize all attention heads
fig, axes = plt.subplots(1, num_heads, figsize=(16, 4))

for h in range(num_heads):
    sns.heatmap(head_weights[h], annot=True, fmt='.2f', cmap='viridis',
                xticklabels=tokens, yticklabels=tokens, ax=axes[h],
                vmin=0, vmax=1, linewidths=0.5, cbar=h == num_heads - 1)
    axes[h].set_title(f'Head {h + 1}', fontsize=12)
    if h == 0:
        axes[h].set_ylabel('Query', fontsize=10)
    axes[h].set_xlabel('Key', fontsize=10)

plt.suptitle('Multi-Head Attention: Each Head Learns Different Patterns', 
             fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print("Notice how each head has different attention patterns.")
print("In trained models, different heads learn to capture different linguistic relationships:")
print("  - Some heads focus on adjacent tokens (local patterns)")
print("  - Some heads focus on syntactic relationships (subject-verb)")
print("  - Some heads focus on coreference (pronoun resolution)")

## Key Takeaways

1. **Self-attention** allows every token to attend to every other token, capturing long-range dependencies in O(1) steps (vs O(n) for RNNs).

2. **Scaling** by $\sqrt{d_k}$ prevents dot products from growing too large, which would push softmax into regions with tiny gradients.

3. **Causal masking** (in decoders) ensures tokens can only attend to previous positions, preserving the autoregressive property needed for generation.

4. **Multi-head attention** lets the model jointly attend to information from different representation subspaces, each head specializing in different patterns.

5. The full Transformer layer also includes **residual connections**, **layer normalization**, and a **feed-forward network** after each attention block.